# LungSeg Kaggle Phase 4/6 Standalone

Notebook autocontenido para ejecutar el equivalente de `make pipeline` en la nube sin depender de `src/lungseg`, `configs/` ni de una instalación editable del proyecto. Ejecuta bootstrap, comprobación de Task06, regeneración de splits, Phase 4 en folds, Phase 6 completa y Phase 5 opcional si existe un manifiesto LIDC. Se omiten los tests/QA del Makefile.


In [ ]:
# Bootstrap de dependencias externas. No instala el paquete local del proyecto.
import importlib.util
import subprocess
import sys

CORE_REQUIREMENTS = [
    "monai>=1.5.2",
    "nibabel>=5.0.0",
    "scipy>=1.11.0",
    "scikit-image>=0.22.0",
    "scikit-learn>=1.3.0",
    "pandas>=2.0.0",
    "tqdm>=4.65",
    "matplotlib>=3.7.0",
    "wandb>=0.17",
]

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch>=2.0.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *CORE_REQUIREMENTS])
print("Dependencias principales listas.")


In [ ]:
# Imports, logging y parámetros equivalentes a las variables del Makefile.
from __future__ import annotations

import copy
import csv
import importlib.util
import itertools
import json
import logging
import math
import os
import random
import shutil
import subprocess
import sys
import tarfile
from collections.abc import Iterable, Sequence
from contextlib import nullcontext
from datetime import datetime
from functools import partial
from itertools import pairwise
from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as functional
from monai.data import CacheDataset, DataLoader, Dataset, PersistentDataset
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.networks.nets import DynUNet, SegResNet, UNet
from monai.transforms import (
    Compose,
    CropForegroundd,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandRotate90d,
    RandScaleIntensityd,
    RandShiftIntensityd,
    ScaleIntensityRanged,
    Spacingd,
)
from monai.utils.misc import set_determinism
from scipy import ndimage
from scipy.stats import wilcoxon
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm


def get_logger(name: str = "standalone_lungseg") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s %(name)s :: %(message)s", "%H:%M:%S"))
    logger.addHandler(handler)
    logger.propagate = False
    return logger


def wandb_enabled() -> bool:
    return bool(os.environ.get("WANDB_API_KEY"))


LOGGER = get_logger()
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working" if IS_KAGGLE else ".")).resolve()
RUN_ID = os.environ.get("RUN_ID", datetime.now().strftime("%Y%m%d-%H%M%S"))
OUTPUTS_ROOT = Path(os.environ.get("OUTPUTS_ROOT", str(WORK_DIR / "outputs" / "full-pipeline" / RUN_ID))).resolve()
SPLITS_DIR = Path(os.environ.get("SPLITS_DIR", str(WORK_DIR / "data" / "splits"))).resolve()
CACHE_DIR = Path(os.environ.get("MONAI_CACHE_DIR", str(WORK_DIR / "data" / "cache" / "monai"))).resolve()


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_optional_int(name: str) -> int | None:
    value = os.environ.get(name, "").strip()
    return int(value) if value else None


def env_list(name: str, default: Sequence[Any], cast=str) -> list[Any]:
    value = os.environ.get(name, "").strip()
    if not value:
        return list(default)
    return [cast(item) for item in value.split()]


SEED = env_int("SEED", 42)
N_FOLDS = env_int("N_FOLDS", 5)
FOLDS = env_list("FOLDS", list(range(N_FOLDS)), int)
PHASE4_FOLDS = env_list("PHASE4_FOLDS", FOLDS, int)
PHASE6_FOLDS = env_list("PHASE6_FOLDS", FOLDS, int)

# En la nube el perfil por defecto es kaggle_p100. Para copiar literalmente el Makefile local, usa TRAINING=local_5060.
TRAINING_PROFILE = os.environ.get("TRAINING", "kaggle_p100")
MODEL_NAME = os.environ.get("MODEL", "segresnet_lung")

PHASE4_MAX_ITER = env_optional_int("PHASE4_MAX_ITER")
PHASE4_VAL_EVERY = env_optional_int("PHASE4_VAL_EVERY")
PHASE4_PATIENCE = env_optional_int("PHASE4_PATIENCE")
PHASE4_CACHE_RATE = env_float("PHASE4_CACHE_RATE", 1.0)
PHASE4_CACHE_WORKERS = env_int("PHASE4_CACHE_WORKERS", 2)
PHASE4_NUM_WORKERS = env_int("PHASE4_NUM_WORKERS", 2)
PHASE4_PIN_MEMORY = env_bool("PHASE4_PIN_MEMORY", True)

PHASE6_MAX_ITER = env_optional_int("PHASE6_MAX_ITER")
PHASE6_VAL_EVERY = env_optional_int("PHASE6_VAL_EVERY")
PHASE6_PATIENCE = env_optional_int("PHASE6_PATIENCE")
FRACTIONS = env_list("FRACTIONS", [0.25, 0.5, 1.0], float)
AUGS = env_list("AUGS", ["none", "standard"], str)
SEEDS = env_list("SEEDS", [0, 1, 2], int)

RUN_PHASE4 = env_bool("RUN_PHASE4", True)
RUN_PHASE6 = env_bool("RUN_PHASE6", True)
RUN_PHASE5 = env_bool("RUN_PHASE5", True)
PHASE5_E2E = env_bool("PHASE5_E2E", False)
LIDC_MANIFEST = Path(os.environ.get("LIDC_MANIFEST", str(WORK_DIR / "data" / "processed" / "lidc" / "nodule_manifest.csv"))).resolve()

print(f"WORK_DIR={WORK_DIR}")
print(f"OUTPUTS_ROOT={OUTPUTS_ROOT}")
print(f"TRAINING={TRAINING_PROFILE} MODEL={MODEL_NAME}")
print(f"PHASE4_FOLDS={PHASE4_FOLDS} PHASE6_FOLDS={PHASE6_FOLDS}")
print(f"FRACTIONS={FRACTIONS} AUGS={AUGS} SEEDS={SEEDS}")


In [ ]:
# Configuración embebida: reemplaza Hydra/configs/*.yaml.
class Cfg(dict):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.update(*args, **kwargs)

    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError as exc:
            raise AttributeError(key) from exc

    def __setattr__(self, key, value):
        self[key] = value

    def __setitem__(self, key, value):
        super().__setitem__(key, cfgify(value))

    def update(self, *args, **kwargs):
        for key, value in dict(*args, **kwargs).items():
            self[key] = value


def cfgify(value):
    if isinstance(value, Cfg):
        return value
    if isinstance(value, dict):
        return Cfg(value)
    if isinstance(value, list):
        return [cfgify(item) for item in value]
    return value


def to_plain(value):
    if isinstance(value, Cfg):
        return {key: to_plain(item) for key, item in value.items()}
    if isinstance(value, dict):
        return {key: to_plain(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_plain(item) for item in value]
    if isinstance(value, tuple):
        return [to_plain(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    return value


def select(cfg: Cfg | dict, key: str, default=None):
    current = cfg
    for part in key.split("."):
        if isinstance(current, dict) and part in current:
            current = current[part]
        else:
            return default
    return current


def update_cfg(cfg: Cfg, key: str, value, force_add: bool = True) -> None:
    current = cfg
    parts = key.split(".")
    for part in parts[:-1]:
        if part not in current:
            if not force_add:
                raise KeyError(key)
            current[part] = Cfg()
        current = current[part]
    current[parts[-1]] = value


DATA_TASK06 = {
    "name": "task06",
    "root": "",
    "dataset_json": "",
    "n_folds": 5,
    "splits_dir": str(SPLITS_DIR),
    "target_spacing": [0.79, 0.79, 1.24],
    "hu_clip": {"a_min": -1024, "a_max": 400, "b_min": 0.0, "b_max": 1.0, "clip": True},
    "crop_foreground": {"source_key": "image", "threshold": 0.1},
    "sampler": {"pos": 2, "neg": 1, "num_samples": 4},
    "cache": {"mode": "disk", "rate": 1.0, "num_workers": 2, "disk_dir": str(CACHE_DIR)},
}

DATA_LIDC = {
    "name": "lidc",
    "root": str(WORK_DIR / "data" / "raw" / "LIDC-IDRI"),
    "manifest": str(LIDC_MANIFEST),
    "pred_masks_dir": str(WORK_DIR / "data" / "processed" / "lidc" / "pred_masks"),
    "n_folds": 5,
    "malignancy_consensus": {"malignant_min": 4, "benign_max": 2},
}

MODEL_CONFIGS = {
    "segresnet_lung": {
        "name": "segresnet",
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 2,
        "init_filters": 16,
        "blocks_down": [1, 2, 2, 4],
        "blocks_up": [1, 1, 1],
        "dropout_prob": 0.2,
        "norm": "instance",
        "act": "relu",
        "loss": {
            "name": "dice_ce",
            "to_onehot_y": True,
            "softmax": True,
            "include_background": False,
            "lambda_dice": 1.0,
            "lambda_ce": 1.0,
        },
    },
    "dynunet_lung": {
        "name": "dynunet",
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 2,
        "kernel_size": [[3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3]],
        "strides": [[1, 1, 1], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 1]],
        "upsample_kernel_size": [[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 1]],
        "norm_name": "instance",
        "deep_supervision": True,
        "deep_supr_num": 3,
        "res_block": True,
        "loss": {
            "name": "dice_ce",
            "to_onehot_y": True,
            "softmax": True,
            "include_background": False,
            "lambda_dice": 1.0,
            "lambda_ce": 1.0,
        },
    },
}

TRAINING_CONFIGS = {
    "kaggle_p100": {
        "name": "kaggle_p100",
        "patch_size": [96, 192, 160],
        "batch_size": 2,
        "num_workers": 2,
        "amp": True,
        "pin_memory": True,
        "optimizer": {"name": "adamw", "lr": 1.0e-4, "weight_decay": 1.0e-5},
        "scheduler": {"name": "poly", "exp": 0.9},
        "augment_regime": "standard",
        "inference": {"sw_batch_size": 4, "overlap": 0.5, "mode": "gaussian", "padding_mode": "constant"},
    },
    "local_5060": {
        "name": "local_5060",
        "patch_size": [96, 96, 96],
        "batch_size": 2,
        "num_workers": 2,
        "amp": True,
        "pin_memory": True,
        "optimizer": {"name": "adamw", "lr": 1.0e-4, "weight_decay": 1.0e-5},
        "scheduler": {"name": "poly", "exp": 0.9},
        "augment_regime": "standard",
        "inference": {"sw_batch_size": 4, "overlap": 0.5, "mode": "gaussian", "padding_mode": "constant"},
    },
    "sanity": {
        "name": "sanity",
        "patch_size": [96, 96, 96],
        "batch_size": 2,
        "num_workers": 0,
        "amp": False,
        "pin_memory": False,
        "optimizer": {"name": "adamw", "lr": 3.0e-4, "weight_decay": 0.0},
        "scheduler": {"name": "poly", "exp": 0.9},
        "augment_regime": "none",
        "inference": {"sw_batch_size": 1, "overlap": 0.25, "mode": "gaussian", "padding_mode": "constant"},
        "sanity": {"overfit_one_batch": True, "max_iterations": 200, "val_every": 50},
    },
}

EXPERIMENT_CONFIGS = {
    "phase4_full": {
        "name": "phase4_full",
        "max_iterations": 50000,
        "val_every": 500,
        "patience": 20,
        "grad_accum_steps": 1,
        "log_every": 50,
    },
    "phase6_ablation": {
        "name": "phase6_ablation",
        "max_iterations": 10000,
        "val_every": 200,
        "patience": 10,
        "grad_accum_steps": 1,
        "log_every": 50,
        "fixed_iterations": True,
        "sweep": {"data_fraction": [0.25, 0.5, 1.0], "aug_regime": ["none", "standard"], "seed": [0, 1, 2]},
    },
}


def make_cfg(
    *,
    experiment_name: str,
    fold: int = 0,
    outputs: str | Path | None = None,
    seed: int = SEED,
    training_profile: str = TRAINING_PROFILE,
    model_name: str = MODEL_NAME,
    data_name: str = "task06",
) -> Cfg:
    if model_name not in MODEL_CONFIGS:
        raise ValueError(f"Modelo desconocido: {model_name}. Opciones: {sorted(MODEL_CONFIGS)}")
    if training_profile not in TRAINING_CONFIGS:
        raise ValueError(f"Perfil de entrenamiento desconocido: {training_profile}. Opciones: {sorted(TRAINING_CONFIGS)}")
    if experiment_name not in EXPERIMENT_CONFIGS:
        raise ValueError(f"Experimento desconocido: {experiment_name}. Opciones: {sorted(EXPERIMENT_CONFIGS)}")

    data = copy.deepcopy(DATA_LIDC if data_name == "lidc" else DATA_TASK06)
    cfg = Cfg(
        {
            "seed": int(seed),
            "fold": int(fold),
            "data": data,
            "model": copy.deepcopy(MODEL_CONFIGS[model_name]),
            "training": copy.deepcopy(TRAINING_CONFIGS[training_profile]),
            "experiment": copy.deepcopy(EXPERIMENT_CONFIGS[experiment_name]),
            "paths": {"outputs": str(outputs or (OUTPUTS_ROOT / experiment_name)), "splits": str(SPLITS_DIR)},
        }
    )

    if experiment_name == "phase4_full":
        cfg.data.cache.rate = PHASE4_CACHE_RATE
        cfg.data.cache.num_workers = PHASE4_CACHE_WORKERS
        cfg.training.num_workers = PHASE4_NUM_WORKERS
        cfg.training.pin_memory = PHASE4_PIN_MEMORY
        if PHASE4_MAX_ITER is not None:
            cfg.experiment.max_iterations = PHASE4_MAX_ITER
        if PHASE4_VAL_EVERY is not None:
            cfg.experiment.val_every = PHASE4_VAL_EVERY
        if PHASE4_PATIENCE is not None:
            cfg.experiment.patience = PHASE4_PATIENCE

    if experiment_name == "phase6_ablation":
        if PHASE6_MAX_ITER is not None:
            cfg.experiment.max_iterations = PHASE6_MAX_ITER
        if PHASE6_VAL_EVERY is not None:
            cfg.experiment.val_every = PHASE6_VAL_EVERY
        if PHASE6_PATIENCE is not None:
            cfg.experiment.patience = PHASE6_PATIENCE

    return cfg


In [ ]:
# Descubrimiento/comprobación de Task06 y regeneración de splits patient-level.
def _safe_extract_tar(tar_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path) as handle:
        dest_resolved = destination.resolve()
        for member in handle.getmembers():
            target = (destination / member.name).resolve()
            if not str(target).startswith(str(dest_resolved)):
                raise RuntimeError(f"Entrada insegura en tar: {member.name}")
        handle.extractall(destination)


def locate_task06_root() -> Path:
    env_root = os.environ.get("TASK06_ROOT", "").strip()
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            WORK_DIR / "data" / "raw" / "Task06_Lung",
            WORK_DIR / "Task06_Lung",
            Path.cwd() / "data" / "raw" / "Task06_Lung",
            Path.cwd() / "Task06_Lung",
        ]
    )
    for candidate in candidates:
        if (candidate / "dataset.json").exists():
            return candidate.resolve()

    tar_candidates = []
    env_tar = os.environ.get("TASK06_TAR", "").strip()
    if env_tar:
        tar_candidates.append(Path(env_tar))
    tar_candidates.extend(
        [
            WORK_DIR / "data" / "raw" / "Task06_Lung.tar",
            WORK_DIR / "Task06_Lung.tar",
            Path.cwd() / "data" / "raw" / "Task06_Lung.tar",
            Path.cwd() / "Task06_Lung.tar",
        ]
    )
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        tar_candidates.extend(sorted(input_dir.rglob("Task06_Lung.tar")))

    for tar_path in tar_candidates:
        if tar_path.exists():
            extract_base = WORK_DIR / "data" / "raw"
            print(f"Extrayendo {tar_path} en {extract_base}...")
            _safe_extract_tar(tar_path, extract_base)
            extracted = extract_base / "Task06_Lung"
            if (extracted / "dataset.json").exists():
                return extracted.resolve()

    if input_dir.exists():
        for dataset_json in sorted(input_dir.rglob("dataset.json")):
            root = dataset_json.parent
            if root.name == "Task06_Lung" and (root / "imagesTr").exists() and (root / "labelsTr").exists():
                return root.resolve()

    raise FileNotFoundError(
        "No encuentro Task06_Lung. En Kaggle añade el dataset con la carpeta Task06_Lung "
        "o define TASK06_ROOT/TASK06_TAR antes de ejecutar el notebook."
    )


def check_task06(root: Path) -> Path:
    dataset_json = root / "dataset.json"
    if not dataset_json.exists():
        raise FileNotFoundError(f"Falta {dataset_json}")
    if not (root / "imagesTr").is_dir():
        raise FileNotFoundError(f"Falta {root / 'imagesTr'}")
    if not (root / "labelsTr").is_dir():
        raise FileNotFoundError(f"Falta {root / 'labelsTr'}")
    images = sorted((root / "imagesTr").glob("*.nii.gz"))
    labels = sorted((root / "labelsTr").glob("*.nii.gz"))
    print(f"Task06 OK: {len(images)} imágenes de train, {len(labels)} etiquetas. Root={root}")
    if not labels:
        raise RuntimeError(f"No hay etiquetas en {root / 'labelsTr'}")
    return dataset_json


def _resolve_dataset_path(path_in_json: str, dataset_dir: Path) -> Path:
    return (dataset_dir / path_in_json.lstrip("./")).resolve()


def _compute_tumor_volume(label_abs: Path) -> float:
    img = nib.load(str(label_abs))
    voxel_volume = float(np.prod(img.header.get_zooms()[:3]))
    data = np.asarray(img.dataobj)
    n_voxels = float((data > 0).sum())
    return n_voxels * voxel_volume


def _assign_strata(volumes: np.ndarray) -> np.ndarray:
    edges = np.percentile(volumes, [100 / 3.0, 200 / 3.0])
    return np.digitize(volumes, edges).astype(int)


def make_splits(dataset_json: Path, out_dir: Path, seed: int = 42, k: int = 5) -> list[Path]:
    dataset_json = Path(dataset_json)
    out_dir = Path(out_dir)
    dataset_dir = dataset_json.parent
    contents = json.loads(dataset_json.read_text(encoding="utf-8"))

    images: list[str] = []
    labels: list[str] = []
    patient_ids: list[str] = []
    volumes: list[float] = []

    for entry in tqdm(contents["training"], desc="Calculando volumen tumoral"):
        image_abs = _resolve_dataset_path(entry["image"], dataset_dir)
        label_abs = _resolve_dataset_path(entry["label"], dataset_dir)
        patient = Path(entry["image"]).name.replace(".nii.gz", "")
        images.append(str(image_abs))
        labels.append(str(label_abs))
        patient_ids.append(patient)
        volumes.append(_compute_tumor_volume(label_abs))

    volumes_arr = np.asarray(volumes, dtype=np.float64)
    strata = _assign_strata(volumes_arr)
    cases = [
        {
            "image": images[i],
            "label": labels[i],
            "patient_id": patient_ids[i],
            "tumor_volume_mm3": round(float(volumes_arr[i]), 3),
            "stratum": int(strata[i]),
        }
        for i in range(len(images))
    ]

    splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_paths: list[Path] = []
    dummy_x = np.zeros(len(cases))

    for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(dummy_x, y=strata, groups=np.asarray(patient_ids))):
        train = sorted([cases[i] for i in train_idx], key=lambda c: c["patient_id"])
        val = sorted([cases[i] for i in val_idx], key=lambda c: c["patient_id"])
        payload = {
            "fold": fold_idx,
            "seed": seed,
            "k": k,
            "n_train": len(train),
            "n_val": len(val),
            "train": train,
            "val": val,
        }
        out_path = out_dir / f"fold_{fold_idx}.json"
        out_path.write_text(json.dumps(payload, indent=2, sort_keys=False) + "\n", encoding="utf-8")
        out_paths.append(out_path)

    return out_paths


def check_splits(splits_dir: Path, folds: Sequence[int]) -> None:
    missing = [fold for fold in folds if not (splits_dir / f"fold_{fold}.json").exists()]
    if missing:
        raise FileNotFoundError(f"Faltan splits para folds: {missing}")
    print(f"Splits OK: {list(folds)} en {splits_dir}")


TASK06_ROOT = locate_task06_root()
TASK06_JSON = check_task06(TASK06_ROOT)
DATA_TASK06["root"] = str(TASK06_ROOT)
DATA_TASK06["dataset_json"] = str(TASK06_JSON)
DATA_TASK06["splits_dir"] = str(SPLITS_DIR)

split_paths = make_splits(TASK06_JSON, SPLITS_DIR, seed=SEED, k=N_FOLDS)
print("Splits generados:")
for path in split_paths:
    print(" -", path)
check_splits(SPLITS_DIR, FOLDS)


In [ ]:
# Transformaciones, DataLoaders, modelos, pérdidas, métricas y entrenador.
KEYS = ["image", "label"]
CROP_METADATA_KEYS = ["foreground_start_coord", "foreground_end_coord"]
AUG_PROB = {"none": 0.0, "standard": 0.15, "aggressive": 0.30}
CACHE_MODE_ALIASES = {
    "0": "none",
    "false": "none",
    "off": "none",
    "no": "none",
    "none": "none",
    "uncached": "none",
    "auto": "auto",
    "1": "ram",
    "true": "ram",
    "cache": "ram",
    "cached": "ram",
    "memory": "ram",
    "ram": "ram",
    "persistent": "disk",
    "persistentdataset": "disk",
    "disk": "disk",
}


def set_global_determinism(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_determinism(seed=seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def _check_no_lr_flip(transforms: list) -> None:
    for tr in transforms:
        if isinstance(tr, RandFlipd):
            axis = tr.flipper.spatial_axis
            if axis is None:
                raise ValueError("RandFlipd sin spatial_axis voltearía todos los ejes, incluido LR.")
            axes = (axis,) if isinstance(axis, int) else tuple(axis)
            if 0 in axes:
                raise ValueError("RandFlipd spatial_axis=0 (LR) está prohibido en CT de tórax; usa spatial_axis=2.")


def _pre_transforms(cfg: Cfg, with_label: bool = True) -> list:
    keys = list(KEYS) if with_label else ["image"]
    spacing_modes = ("bilinear", "nearest") if with_label else ("bilinear",)
    return [
        LoadImaged(keys=keys),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes="RAS"),
        Spacingd(keys=keys, pixdim=tuple(cfg.data.target_spacing), mode=spacing_modes),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=float(cfg.data.hu_clip.a_min),
            a_max=float(cfg.data.hu_clip.a_max),
            b_min=float(cfg.data.hu_clip.b_min),
            b_max=float(cfg.data.hu_clip.b_max),
            clip=bool(cfg.data.hu_clip.clip),
        ),
        CropForegroundd(
            keys=keys,
            source_key="image",
            select_fn=lambda x, t=float(cfg.data.crop_foreground.threshold): x > t,
            allow_smaller=True,
        ),
    ]


def _augmentations(prob: float) -> list:
    if prob <= 0.0:
        return []
    return [
        RandFlipd(keys=KEYS, prob=prob, spatial_axis=2),
        RandRotate90d(keys=KEYS, prob=prob, max_k=3, spatial_axes=(0, 1)),
        RandGaussianNoised(keys=["image"], prob=prob, mean=0.0, std=0.02),
        RandGaussianSmoothd(keys=["image"], prob=prob, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),
        RandScaleIntensityd(keys=["image"], factors=0.10, prob=prob),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=prob),
    ]


def build_train_transforms(cfg: Cfg) -> Compose:
    regime = str(cfg.training.augment_regime)
    if regime not in AUG_PROB:
        raise ValueError(f"augment_regime desconocido: {regime!r}; esperado {list(AUG_PROB)}")
    pre = _pre_transforms(cfg)
    crop = RandCropByPosNegLabeld(
        keys=KEYS,
        label_key="label",
        spatial_size=tuple(cfg.training.patch_size),
        pos=float(cfg.data.sampler.pos),
        neg=float(cfg.data.sampler.neg),
        num_samples=int(cfg.data.sampler.num_samples),
        image_key="image",
        image_threshold=0.0,
        allow_smaller=True,
    )
    transforms = [*pre, crop, *_augmentations(AUG_PROB[regime]), EnsureTyped(keys=[*KEYS, *CROP_METADATA_KEYS], allow_missing_keys=True)]
    _check_no_lr_flip(transforms)
    return Compose(transforms)


def build_val_transforms(cfg: Cfg, with_label: bool = True) -> Compose:
    keys = list(KEYS) if with_label else ["image"]
    return Compose([*_pre_transforms(cfg, with_label=with_label), EnsureTyped(keys=[*keys, *CROP_METADATA_KEYS], allow_missing_keys=True)])


def _resolve_record_path(value: str | Path) -> str:
    path = Path(value)
    return str(path if path.is_absolute() else WORK_DIR / path)


def _load_fold(splits_dir: Path, fold: int) -> tuple[list[dict], list[dict]]:
    fold_path = splits_dir / f"fold_{fold}.json"
    payload = json.loads(fold_path.read_text(encoding="utf-8"))

    def absolutize(records: list[dict]) -> list[dict]:
        return [
            {"image": _resolve_record_path(r["image"]), "label": _resolve_record_path(r["label"]), "patient_id": r["patient_id"], "stratum": int(r.get("stratum", 0))}
            for r in records
        ]

    return absolutize(payload["train"]), absolutize(payload["val"])


def _cache_mode(cfg: Cfg, cache_rate: float) -> str:
    raw_mode = select(cfg, "data.cache.mode", "auto")
    mode = CACHE_MODE_ALIASES.get(str(raw_mode).lower().replace("_", "").replace("-", ""))
    if mode is None:
        raise ValueError(f"data.cache.mode desconocido: {raw_mode!r}")
    if mode == "auto":
        return "ram" if cache_rate > 0.0 else "none"
    return mode


def _resolve_cache_dir(cfg: Cfg, fold: int) -> Path:
    raw_dir = select(cfg, "data.cache.disk_dir", str(CACHE_DIR))
    cache_dir = Path(str(raw_dir))
    if not cache_dir.is_absolute():
        cache_dir = WORK_DIR / cache_dir
    return cache_dir / f"fold_{fold}"


def _plain_datasets(train_files: list[dict], val_files: list[dict], train_tf, val_tf) -> tuple[Dataset, Dataset]:
    return Dataset(data=train_files, transform=train_tf), Dataset(data=val_files, transform=val_tf)


def build_loaders(cfg: Cfg, fold: int) -> tuple[DataLoader, DataLoader]:
    splits_dir = Path(str(select(cfg, "paths.splits", str(SPLITS_DIR))))
    train_files, val_files = _load_fold(splits_dir, fold)
    train_tf = build_train_transforms(cfg)
    val_tf = build_val_transforms(cfg)

    cache_rate = float(select(cfg, "data.cache.rate", 0.0) or 0.0)
    cache_workers = int(select(cfg, "data.cache.num_workers", 0) or 0)
    cache_mode = _cache_mode(cfg, cache_rate)

    if cache_mode == "none":
        train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)
    elif cache_mode == "disk":
        cache_dir = _resolve_cache_dir(cfg, fold)
        try:
            train_cache_dir = cache_dir / "train"
            val_cache_dir = cache_dir / "val"
            train_cache_dir.mkdir(parents=True, exist_ok=True)
            val_cache_dir.mkdir(parents=True, exist_ok=True)
            train_ds = PersistentDataset(data=train_files, transform=train_tf, cache_dir=train_cache_dir)
            val_ds = PersistentDataset(data=val_files, transform=val_tf, cache_dir=val_cache_dir)
            LOGGER.info("Usando PersistentDataset en %s", cache_dir)
        except OSError as exc:
            LOGGER.warning("Cache en disco no disponible en %s (%s); usando Dataset sin caché.", cache_dir, exc)
            train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)
    elif cache_rate > 0.0:
        try:
            train_ds = CacheDataset(data=train_files, transform=train_tf, cache_rate=cache_rate, num_workers=cache_workers, copy_cache=False)
            val_ds = CacheDataset(data=val_files, transform=val_tf, cache_rate=cache_rate, num_workers=cache_workers, copy_cache=False)
            LOGGER.info("Usando CacheDataset en RAM con cache_rate=%.3f", cache_rate)
        except PermissionError as exc:
            LOGGER.warning("CacheDataset no disponible (%s); usando Dataset sin caché.", exc)
            train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)
    else:
        train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg.training.batch_size),
        shuffle=True,
        num_workers=int(cfg.training.num_workers),
        pin_memory=bool(cfg.training.pin_memory),
        worker_init_fn=seed_worker,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=int(cfg.training.num_workers),
        pin_memory=bool(cfg.training.pin_memory),
        worker_init_fn=seed_worker,
    )
    return train_loader, val_loader


def build_segresnet(cfg: Cfg):
    model_cfg = cfg.model if "model" in cfg else cfg
    if bool(model_cfg.get("deep_supervision", False)):
        raise ValueError("SegResNet no soporta deep_supervision en MONAI 1.5.x")
    return SegResNet(
        spatial_dims=int(model_cfg.get("spatial_dims", 3)),
        in_channels=int(model_cfg.get("in_channels", 1)),
        out_channels=int(model_cfg.get("out_channels", 2)),
        init_filters=int(model_cfg.get("init_filters", 16)),
        blocks_down=tuple(model_cfg.get("blocks_down", [1, 2, 2, 4])),
        blocks_up=tuple(model_cfg.get("blocks_up", [1, 1, 1])),
        dropout_prob=float(model_cfg.get("dropout_prob", 0.0)),
        norm=str(model_cfg.get("norm", "instance")).upper(),
        act=str(model_cfg.get("act", "relu")).upper(),
    )


def build_dynunet(cfg: Cfg):
    model_cfg = cfg.model if "model" in cfg else cfg
    filters = model_cfg.get("filters", None)
    return DynUNet(
        spatial_dims=int(model_cfg.get("spatial_dims", 3)),
        in_channels=int(model_cfg.get("in_channels", 1)),
        out_channels=int(model_cfg.get("out_channels", 2)),
        kernel_size=[tuple(k) for k in model_cfg.kernel_size],
        strides=[tuple(s) for s in model_cfg.strides],
        upsample_kernel_size=[tuple(k) for k in model_cfg.upsample_kernel_size],
        filters=None if filters is None else list(filters),
        dropout=model_cfg.get("dropout", None),
        norm_name=str(model_cfg.get("norm_name", "instance")).upper(),
        deep_supervision=bool(model_cfg.get("deep_supervision", False)),
        deep_supr_num=int(model_cfg.get("deep_supr_num", 1)),
        res_block=bool(model_cfg.get("res_block", True)),
    )


def build_unet(cfg: Cfg):
    model_cfg = cfg.model if "model" in cfg else cfg
    return UNet(
        spatial_dims=int(model_cfg.get("spatial_dims", 3)),
        in_channels=int(model_cfg.get("in_channels", 1)),
        out_channels=int(model_cfg.get("out_channels", 2)),
        channels=tuple(model_cfg.get("channels", [16, 32, 64, 128, 256])),
        strides=tuple(model_cfg.get("strides", [2, 2, 2, 2])),
        num_res_units=int(model_cfg.get("num_res_units", 2)),
        dropout=float(model_cfg.get("dropout", 0.0)),
        norm=str(model_cfg.get("norm", "instance")).upper(),
        act=str(model_cfg.get("act", "prelu")).upper(),
    )


def build_model(cfg: Cfg) -> torch.nn.Module:
    model_cfg = cfg.model if "model" in cfg else cfg
    name = str(model_cfg.get("name", "segresnet")).lower()
    builders = {"segresnet": build_segresnet, "dynunet": build_dynunet, "unet": build_unet, "unet_baseline": build_unet}
    if name not in builders:
        raise ValueError(f"model.name desconocido: {name!r}; esperado {sorted(builders)}")
    return builders[name](cfg)


def build_loss(cfg: Cfg) -> torch.nn.Module:
    loss_cfg = cfg.model.loss if "model" in cfg and "loss" in cfg.model else cfg.loss
    name = str(loss_cfg.get("name", "dice_ce")).lower()
    if name != "dice_ce":
        raise ValueError(f"loss.name desconocido: {name!r}; solo se soporta dice_ce")
    return DiceCELoss(
        include_background=bool(loss_cfg.get("include_background", False)),
        to_onehot_y=bool(loss_cfg.get("to_onehot_y", True)),
        sigmoid=bool(loss_cfg.get("sigmoid", False)),
        softmax=bool(loss_cfg.get("softmax", True)),
        lambda_dice=float(loss_cfg.get("lambda_dice", 1.0)),
        lambda_ce=float(loss_cfg.get("lambda_ce", 1.0)),
    )


def _normalized_weights(n_outputs: int, weights: list[float] | None) -> torch.Tensor:
    if n_outputs <= 0:
        raise ValueError("La pila de deep supervision debe contener al menos una salida")
    raw = weights if weights is not None else [0.5 ** (i + 1) for i in range(n_outputs)]
    if len(raw) != n_outputs:
        raise ValueError(f"Se esperaban {n_outputs} pesos de deep supervision, se obtuvieron {len(raw)}")
    tensor = torch.as_tensor(raw, dtype=torch.float32)
    if torch.any(tensor < 0) or float(tensor.sum()) <= 0.0:
        raise ValueError("Los pesos de deep supervision deben ser no negativos y sumar > 0")
    return tensor / tensor.sum()


def deep_supervision_loss(out_stacked: torch.Tensor, target: torch.Tensor, base_loss, weights: list[float] | None = None) -> torch.Tensor:
    if out_stacked.dim() == target.dim() + 1:
        n_outputs = int(out_stacked.shape[1])
        norm_weights = _normalized_weights(n_outputs, weights).to(device=out_stacked.device, dtype=out_stacked.dtype)
        total = out_stacked.new_tensor(0.0)
        for idx in range(n_outputs):
            pred = out_stacked[:, idx]
            label = target
            if tuple(label.shape[2:]) != tuple(pred.shape[2:]):
                label = functional.interpolate(label.float(), size=pred.shape[2:], mode="nearest")
                if target.dtype in (torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64):
                    label = label.to(dtype=target.dtype)
            total = total + norm_weights[idx] * base_loss(pred, label)
        return total
    return base_loss(out_stacked, target)


def build_poly_scheduler(optimizer: torch.optim.Optimizer, max_steps: int, exp: float = 0.9) -> LambdaLR:
    if max_steps <= 0:
        raise ValueError("max_steps debe ser > 0")

    def lr_lambda(step: int) -> float:
        clamped = min(max(int(step), 0), int(max_steps))
        return (1.0 - clamped / float(max_steps)) ** float(exp)

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


def predict_volume(model: torch.nn.Module, image: torch.Tensor, cfg: Cfg) -> torch.Tensor:
    def predictor(window: torch.Tensor) -> torch.Tensor:
        out = model(window)
        if isinstance(out, (tuple, list)):
            out = out[0]
        if isinstance(out, torch.Tensor) and out.dim() == window.dim() + 1:
            out = out[:, 0]
        return out

    inference_cfg = cfg.training.get("inference", {})
    return sliding_window_inference(
        inputs=image,
        roi_size=tuple(int(v) for v in cfg.training.patch_size),
        sw_batch_size=int(inference_cfg.get("sw_batch_size", 1)),
        predictor=predictor,
        overlap=float(inference_cfg.get("overlap", 0.25)),
        mode=str(inference_cfg.get("mode", "gaussian")),
        padding_mode=str(inference_cfg.get("padding_mode", "constant")),
    )


def _to_binary_array(value: torch.Tensor | np.ndarray) -> np.ndarray:
    arr = value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)
    if arr.ndim >= 5 and arr.shape[1] > 1:
        arr = np.argmax(arr, axis=1, keepdims=True)
    elif arr.ndim >= 5 and arr.shape[1] == 1:
        arr = arr > 0.5
    elif arr.ndim == 4:
        arr = arr[:, None] > 0.5
    return np.asarray(arr).astype(bool)


def _surface(mask: np.ndarray) -> np.ndarray:
    if not mask.any():
        return mask.astype(bool)
    eroded = ndimage.binary_erosion(mask)
    return np.logical_xor(mask, eroded)


def _hd95_one(pred: np.ndarray, label: np.ndarray, spacing: Sequence[float] | None) -> float:
    if not pred.any() and not label.any():
        return 0.0
    if not pred.any() or not label.any():
        return float("nan")
    pred_surface = _surface(pred)
    label_surface = _surface(label)
    sampling = None if spacing is None else tuple(float(s) for s in spacing)
    pred_to_label = ndimage.distance_transform_edt(~label_surface, sampling=sampling)[pred_surface]
    label_to_pred = ndimage.distance_transform_edt(~pred_surface, sampling=sampling)[label_surface]
    return float(np.percentile(np.concatenate([pred_to_label, label_to_pred]), 95))


def compute_segmentation_metrics(prediction: torch.Tensor | np.ndarray, label: torch.Tensor | np.ndarray, spacing: Sequence[float] | None = None) -> dict[str, float]:
    pred_bin = _to_binary_array(prediction)
    label_bin = _to_binary_array(label)
    if pred_bin.shape != label_bin.shape:
        raise ValueError(f"prediction/label shape mismatch: {pred_bin.shape} != {label_bin.shape}")
    dice_scores: list[float] = []
    hd95_scores: list[float] = []
    for pred_one, label_one in zip(pred_bin[:, 0], label_bin[:, 0], strict=True):
        intersection = float(np.logical_and(pred_one, label_one).sum())
        denom = float(pred_one.sum() + label_one.sum())
        dice_scores.append(1.0 if denom == 0.0 else 2.0 * intersection / denom)
        hd95_scores.append(_hd95_one(pred_one, label_one, spacing=spacing))
    hd95_arr = np.asarray(hd95_scores, dtype=np.float64)
    return {"dice": float(np.mean(dice_scores)), "hd95": float(np.nanmean(hd95_arr)) if not np.isnan(hd95_arr).all() else float("nan")}


def _outputs_dir(cfg: Cfg) -> Path:
    value = select(cfg, "paths.outputs", str(OUTPUTS_ROOT / "manual"))
    path = Path(str(value))
    path.mkdir(parents=True, exist_ok=True)
    return path


def _autocast_context(device: torch.device, enabled: bool):
    if device.type == "cuda" and enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def _make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def _model_output_for_loss(output: torch.Tensor | tuple | list) -> torch.Tensor:
    if isinstance(output, (tuple, list)):
        output = output[0]
    return output


def _batch_to_device(batch: dict[str, Any], device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    return batch["image"].to(device, non_blocking=True), batch["label"].to(device, non_blocking=True)


def _make_optimizer(cfg: Cfg, model: torch.nn.Module) -> torch.optim.Optimizer:
    opt_cfg = cfg.training.optimizer
    name = str(opt_cfg.get("name", "adamw")).lower()
    if name != "adamw":
        raise ValueError(f"optimizer.name desconocido: {name!r}; solo se soporta adamw")
    return torch.optim.AdamW(model.parameters(), lr=float(opt_cfg.get("lr", 1.0e-4)), weight_decay=float(opt_cfg.get("weight_decay", 1.0e-5)))


def _make_train_iter(train_loader: Iterable, cfg: Cfg) -> Iterable:
    if bool(select(cfg, "training.sanity.overfit_one_batch", False)):
        first_batch = next(iter(train_loader))
        return itertools.repeat(first_batch)
    return itertools.cycle(train_loader)


@torch.no_grad()
def _validate(cfg: Cfg, model: torch.nn.Module, val_loader: Iterable, device: torch.device) -> dict[str, float]:
    model.eval()
    dice_values: list[float] = []
    hd95_values: list[float] = []
    spacing = select(cfg, "data.target_spacing", None)
    for batch in val_loader:
        image, label = _batch_to_device(batch, device)
        logits = predict_volume(model, image, cfg)
        metrics = compute_segmentation_metrics(logits, label, spacing=spacing)
        dice_values.append(metrics["dice"])
        if math.isfinite(metrics["hd95"]):
            hd95_values.append(metrics["hd95"])
    model.train()
    return {"val_dice": float(sum(dice_values) / max(len(dice_values), 1)), "val_hd95": float(sum(hd95_values) / len(hd95_values)) if hd95_values else float("nan")}


def _write_metrics_csv(path: Path, rows: list[dict[str, float | int]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def _finite_or_none(value: float) -> float | None:
    return float(value) if math.isfinite(float(value)) else None


def _save_checkpoint(path: Path, cfg: Cfg, model: torch.nn.Module, optimizer: torch.optim.Optimizer, step: int, metrics: dict[str, float]) -> None:
    torch.save(
        {
            "step": step,
            "metrics": metrics,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "cfg": to_plain(cfg),
        },
        path,
    )


def train_iters(cfg: Cfg, model: torch.nn.Module, loaders: tuple[DataLoader, DataLoader]) -> dict:
    set_global_determinism(int(select(cfg, "seed", 42)))
    train_loader, val_loader = loaders
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.train()

    max_iterations = int(select(cfg, "training.sanity.max_iterations", select(cfg, "experiment.max_iterations", 50000)))
    val_every = int(select(cfg, "training.sanity.val_every", select(cfg, "experiment.val_every", 500)))
    log_every = int(select(cfg, "experiment.log_every", 50))
    grad_accum_steps = max(int(select(cfg, "experiment.grad_accum_steps", 1)), 1)
    patience = int(select(cfg, "experiment.patience", 20))
    fixed_iterations = bool(select(cfg, "experiment.fixed_iterations", False))

    optimizer = _make_optimizer(cfg, model)
    scheduler = build_poly_scheduler(optimizer, max_steps=max_iterations, exp=float(cfg.training.scheduler.get("exp", 0.9)))
    base_loss = build_loss(cfg)
    amp_enabled = bool(cfg.training.get("amp", False)) and device.type == "cuda"
    scaler = _make_grad_scaler(enabled=amp_enabled)

    out_dir = _outputs_dir(cfg)
    ckpt_dir = out_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.csv"
    summary_path = out_dir / "summary.json"
    best_ckpt = ckpt_dir / "best.pt"
    last_ckpt = ckpt_dir / "last.pt"

    wandb_run = None
    if wandb_enabled():
        try:
            import wandb

            wandb_run = wandb.init(project="lungseg", config=to_plain(cfg), dir=str(out_dir))
        except Exception as exc:
            LOGGER.warning("W&B solicitado pero no pudo inicializarse: %s", exc)

    train_iter = iter(_make_train_iter(train_loader, cfg))
    optimizer.zero_grad(set_to_none=True)
    global_step = 0
    micro_step = 0
    running_loss = 0.0
    best_dice = -1.0
    best_hd95 = float("nan")
    best_step = 0
    validations_without_improvement = 0
    rows: list[dict[str, float | int]] = []
    last_metrics = {"val_dice": float("nan"), "val_hd95": float("nan")}

    progress = tqdm(total=max_iterations, initial=global_step, desc=str(out_dir.relative_to(WORK_DIR)) if str(out_dir).startswith(str(WORK_DIR)) else str(out_dir))
    try:
        while global_step < max_iterations:
            batch = next(train_iter)
            image, label = _batch_to_device(batch, device)
            with _autocast_context(device, amp_enabled):
                output = _model_output_for_loss(model(image))
                loss = deep_supervision_loss(output, label, base_loss) / grad_accum_steps

            scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * grad_accum_steps
            micro_step += 1

            if micro_step % grad_accum_steps != 0:
                continue

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            progress.update(1)

            lr = float(optimizer.param_groups[0]["lr"])
            mean_train_loss = running_loss / grad_accum_steps
            running_loss = 0.0

            if global_step % log_every == 0 or global_step == 1:
                LOGGER.info("step=%d/%d loss=%.4f lr=%.6g", global_step, max_iterations, mean_train_loss, lr)
                if wandb_run is not None:
                    wandb_run.log({"train/loss": mean_train_loss, "lr": lr, "step": global_step})

            should_validate = global_step % val_every == 0 or global_step == max_iterations
            if not should_validate:
                continue

            last_metrics = _validate(cfg, model, val_loader, device)
            row = {"step": global_step, "train_loss": mean_train_loss, "lr": lr, "val_dice": last_metrics["val_dice"], "val_hd95": last_metrics["val_hd95"]}
            rows.append(row)
            _write_metrics_csv(metrics_path, rows)
            LOGGER.info(
                "val step=%d dice=%.4f hd95=%s",
                global_step,
                last_metrics["val_dice"],
                f"{last_metrics['val_hd95']:.3f}" if math.isfinite(last_metrics["val_hd95"]) else "nan",
            )
            if wandb_run is not None:
                wandb_run.log({"val/dice": last_metrics["val_dice"], "val/hd95": last_metrics["val_hd95"], "step": global_step})

            if last_metrics["val_dice"] > best_dice:
                best_dice = last_metrics["val_dice"]
                best_hd95 = last_metrics["val_hd95"]
                best_step = global_step
                validations_without_improvement = 0
                _save_checkpoint(best_ckpt, cfg, model, optimizer, global_step, last_metrics)
            else:
                validations_without_improvement += 1

            _save_checkpoint(last_ckpt, cfg, model, optimizer, global_step, last_metrics)
            if not fixed_iterations and validations_without_improvement >= patience:
                LOGGER.info("early stopping en step=%d tras %d validaciones sin mejora", global_step, patience)
                break
    finally:
        progress.close()

    if not rows:
        last_metrics = _validate(cfg, model, val_loader, device)
        rows.append({"step": global_step, "train_loss": float("nan"), "lr": float(optimizer.param_groups[0]["lr"]), "val_dice": last_metrics["val_dice"], "val_hd95": last_metrics["val_hd95"]})
        _write_metrics_csv(metrics_path, rows)
        if last_metrics["val_dice"] > best_dice:
            best_dice = last_metrics["val_dice"]
            best_hd95 = last_metrics["val_hd95"]
            best_step = global_step
            _save_checkpoint(best_ckpt, cfg, model, optimizer, global_step, last_metrics)

    _save_checkpoint(last_ckpt, cfg, model, optimizer, global_step, last_metrics)
    summary = {
        "best_step": int(best_step),
        "best_val_dice": _finite_or_none(best_dice),
        "best_val_hd95": _finite_or_none(best_hd95),
        "last_step": int(global_step),
        "checkpoint_path": str(best_ckpt),
        "last_checkpoint_path": str(last_ckpt),
        "metrics_path": str(metrics_path),
    }
    summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    if wandb_run is not None:
        wandb_run.finish()
    return summary


def train_segmentation_run(cfg: Cfg) -> dict:
    model = build_model(cfg)
    loaders = build_loaders(cfg, fold=int(cfg.fold))
    summary = train_iters(cfg, model, loaders)
    LOGGER.info("training complete: best Dice=%s checkpoint=%s", summary["best_val_dice"], summary["checkpoint_path"])
    return summary


In [ ]:
# Phase 6: muestreo fraccional, ejecución de celdas de ablación y reporte agregado.
def _sample_train_records(records: list[dict], fraction: float, seed: int) -> list[dict]:
    if fraction >= 1.0:
        return list(records)
    if fraction <= 0.0:
        raise ValueError("data_fraction debe ser > 0")
    rng = np.random.default_rng(seed)
    selected: list[dict] = []
    strata = sorted({int(r.get("stratum", 0)) for r in records})
    for stratum in strata:
        bucket = [r for r in records if int(r.get("stratum", 0)) == stratum]
        n_keep = max(1, round(len(bucket) * fraction))
        indices = np.sort(rng.choice(len(bucket), size=min(n_keep, len(bucket)), replace=False))
        selected.extend(bucket[int(i)] for i in indices)
    return sorted(selected, key=lambda r: r["patient_id"])


def _write_fractional_split(cfg: Cfg, fraction: float, seed: int) -> Path:
    fold = int(select(cfg, "fold", 0))
    source_dir = Path(str(select(cfg, "paths.splits", str(SPLITS_DIR))))
    source_path = source_dir / f"fold_{fold}.json"
    payload = json.loads(source_path.read_text(encoding="utf-8"))
    payload["train"] = _sample_train_records(payload["train"], fraction=fraction, seed=seed)
    payload["n_train"] = len(payload["train"])
    payload["ablation"] = {"data_fraction": fraction, "seed": seed}

    out_dir = _outputs_dir(cfg) / "ablation_splits"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"fold_{fold}_frac_{fraction:g}_seed_{seed}.json"
    out_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")

    active_dir = out_dir / f"active_frac_{fraction:g}_seed_{seed}"
    active_dir.mkdir(parents=True, exist_ok=True)
    (active_dir / f"fold_{fold}.json").write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    return active_dir


def run_ablation_cell(cfg: Cfg) -> dict:
    seed = int(select(cfg, "seed", select(cfg, "experiment.seed", 42)))
    fraction = float(select(cfg, "data_fraction", select(cfg, "experiment.data_fraction", 1.0)))
    augment_regime = str(select(cfg, "aug_regime", select(cfg, "training.augment_regime", "standard")))

    cell_cfg = cfgify(copy.deepcopy(to_plain(cfg)))
    update_cfg(cell_cfg, "seed", seed)
    update_cfg(cell_cfg, "training.augment_regime", augment_regime)
    update_cfg(cell_cfg, "experiment.fixed_iterations", True)
    split_dir = _write_fractional_split(cell_cfg, fraction=fraction, seed=seed)
    update_cfg(cell_cfg, "paths.splits", str(split_dir))

    summary = train_segmentation_run(cell_cfg)
    result = {"seed": seed, "data_fraction": fraction, "augment_regime": augment_regime, **summary}
    out_dir = _outputs_dir(cell_cfg) / "ablation"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"frac_{fraction:g}_aug_{augment_regime}_seed_{seed}.json"
    out_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    result["result_path"] = str(out_path)
    return result


def _markdown_table(df: pd.DataFrame) -> list[str]:
    columns = list(df.columns)
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for _, row in df.iterrows():
        values = []
        for col in columns:
            value = row[col]
            values.append(f"{value:.4g}" if isinstance(value, float) else str(value))
        lines.append("| " + " | ".join(values) + " |")
    return lines


def analyze_ablation(outputs_dir: Path) -> Path:
    outputs_dir = Path(outputs_dir)
    ablation_dir = outputs_dir / "ablation"
    rows = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(ablation_dir.glob("*.json"))]
    if not rows:
        raise FileNotFoundError(f"No hay JSON de ablación en {ablation_dir}")

    df = pd.DataFrame(rows)
    summary = (
        df.groupby(["data_fraction", "augment_regime"], as_index=False)
        .agg(
            dice_median=("best_val_dice", "median"),
            dice_q1=("best_val_dice", lambda s: s.quantile(0.25)),
            dice_q3=("best_val_dice", lambda s: s.quantile(0.75)),
            hd95_median=("best_val_hd95", "median"),
            hd95_q1=("best_val_hd95", lambda s: s.quantile(0.25)),
            hd95_q3=("best_val_hd95", lambda s: s.quantile(0.75)),
            n=("best_val_dice", "count"),
        )
        .sort_values(["data_fraction", "augment_regime"])
    )
    summary_path = outputs_dir / "ablation_summary.csv"
    summary.to_csv(summary_path, index=False)

    plot_path = outputs_dir / "ablation_violin.png"
    try:
        import matplotlib.pyplot as plt

        plot_df = df.copy()
        plot_df["cell"] = plot_df["data_fraction"].astype(str) + " / " + plot_df["augment_regime"].astype(str)
        fig, ax = plt.subplots(figsize=(10, 4))
        ordered = sorted(plot_df["cell"].unique())
        ax.violinplot([plot_df.loc[plot_df["cell"] == cell, "best_val_dice"].dropna() for cell in ordered])
        ax.set_xticks(range(1, len(ordered) + 1), ordered, rotation=30, ha="right")
        ax.set_ylabel("Best val Dice")
        ax.set_title("Phase 6 ablation")
        fig.tight_layout()
        fig.savefig(plot_path, dpi=160)
        plt.close(fig)
    except Exception as exc:
        LOGGER.warning("No se pudo generar violin plot: %s", exc)
        plot_path = Path("")

    wilcoxon_rows = []
    for fraction in sorted(df["data_fraction"].unique()):
        sub = df[df["data_fraction"] == fraction]
        pivot = sub.pivot_table(index="seed", columns="augment_regime", values="best_val_dice")
        if {"none", "standard"}.issubset(pivot.columns) and len(pivot.dropna()) >= 2:
            try:
                stat, p_value = wilcoxon(pivot["none"], pivot["standard"])
                wilcoxon_rows.append((fraction, float(stat), float(p_value)))
            except ValueError:
                wilcoxon_rows.append((fraction, float("nan"), float("nan")))

    report_path = outputs_dir / "REPORT_ABLATION.md"
    lines = ["# INFORME_ABLACIÓN", "", "## Resumen", "", *_markdown_table(summary), "", "## Pruebas pareadas de Wilcoxon", ""]
    if wilcoxon_rows:
        lines.append("| data_fraction | statistic | p_value |")
        lines.append("|---:|---:|---:|")
        for fraction, stat, p_value in wilcoxon_rows:
            lines.append(f"| {fraction:g} | {stat:.4g} | {p_value:.4g} |")
    else:
        lines.append("No hay suficientes semillas emparejadas para ejecutar las pruebas de Wilcoxon.")
    lines.extend(["", f"- Summary CSV: `{summary_path}`"])
    if str(plot_path):
        lines.append(f"- Violin plot: `{plot_path}`")
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return report_path


In [ ]:
# Phase 5 opcional: radiómica LIDC/clasificación. Se salta si no existe el manifiesto.
def locate_lidc_manifest() -> Path | None:
    candidates = [LIDC_MANIFEST, WORK_DIR / "data" / "processed" / "lidc" / "nodule_manifest.csv", Path.cwd() / "data" / "processed" / "lidc" / "nodule_manifest.csv"]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        matches = sorted(input_dir.rglob("nodule_manifest.csv"))
        if matches:
            return matches[0].resolve()
    return None


def ensure_phase5_dependencies() -> None:
    if importlib.util.find_spec("radiomics") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/AIM-Harvard/pyradiomics.git", "pylidc>=0.2.3"])
    if importlib.util.find_spec("xgboost") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"])


def extract_radiomics(image_path: Path, mask_path: Path) -> dict[str, float]:
    try:
        from radiomics import featureextractor
    except ImportError as exc:
        raise ImportError("PyRadiomics es necesario para Phase 5.") from exc

    image_path = Path(image_path)
    mask_path = Path(mask_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Imagen radiómica no encontrada: {image_path}")
    if not mask_path.exists():
        raise FileNotFoundError(f"Máscara radiómica no encontrada: {mask_path}")

    extractor = featureextractor.RadiomicsFeatureExtractor(resampledPixelSpacing=[1.0, 1.0, 1.0], interpolator="sitkBSpline", binWidth=25, label=1)
    extractor.enableImageTypes(Original={}, LoG={"sigma": [1.0, 2.0, 3.0]}, Wavelet={})
    result = extractor.execute(str(image_path), str(mask_path))
    features: dict[str, float] = {}
    for key, value in result.items():
        if key.startswith("diagnostics_"):
            continue
        try:
            features[key] = float(value)
        except (TypeError, ValueError):
            continue
    return features


REQUIRED_LIDC_COLUMNS = {"patient_id", "nodule_id", "image", "mask_gt", "malignancy_median"}


def _read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"LIDC manifest no encontrado: {path}")
    if path.suffix.lower() == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows = payload["nodules"] if isinstance(payload, dict) and "nodules" in payload else payload
        return pd.DataFrame(rows)
    return pd.read_csv(path)


def _label_from_malignancy(value: float, benign_max: float, malignant_min: float) -> int | None:
    if value >= malignant_min:
        return 1
    if value <= benign_max:
        return 0
    return None


def _resolve_path(value: object, base_dir: Path) -> Path:
    path = Path(str(value))
    return path if path.is_absolute() else base_dir / path


def _mask_for_row(row: pd.Series, cfg: Cfg, e2e: bool, base_dir: Path) -> Path:
    if not e2e:
        return _resolve_path(row["mask_gt"], base_dir)
    pred_dir_value = select(cfg, "data.pred_masks_dir", None)
    if pred_dir_value is None:
        raise ValueError("e2e=True requiere cfg.data.pred_masks_dir con máscaras predichas")
    pred_dir = Path(str(pred_dir_value))
    candidates = [
        pred_dir / f"{row['patient_id']}_{row['nodule_id']}.nii.gz",
        pred_dir / f"{row['nodule_id']}.nii.gz",
        pred_dir / str(row.get("mask_pred", "")),
    ]
    for candidate in candidates:
        if candidate.name and candidate.exists():
            return candidate
    raise FileNotFoundError(f"Máscara predicha no encontrada para patient_id={row['patient_id']} nodule_id={row['nodule_id']} en {pred_dir}")


def build_radiomic_dataset(cfg: Cfg, e2e: bool = False):
    data_name = str(select(cfg, "data.name", "")).lower()
    if data_name in {"task06", "task06_lung", "msd_task06"}:
        raise ValueError("MSD Task06_Lung no tiene etiquetas benigno/maligno; usa LIDC-IDRI para Phase 5")
    if data_name not in {"lidc", "lidc-idri", "lidc_idri"}:
        raise ValueError(f"Dataset de clasificación no soportado: {data_name!r}")

    manifest_path = Path(str(select(cfg, "data.manifest", "")))
    manifest = _read_manifest(manifest_path)
    missing = sorted(REQUIRED_LIDC_COLUMNS - set(manifest.columns))
    if missing:
        raise ValueError(f"El manifiesto LIDC no tiene columnas requeridas: {missing}")

    benign_max = float(select(cfg, "data.malignancy_consensus.benign_max", 2))
    malignant_min = float(select(cfg, "data.malignancy_consensus.malignant_min", 4))
    rows: list[dict[str, object]] = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Extrayendo radiómica LIDC"):
        label = _label_from_malignancy(float(row["malignancy_median"]), benign_max, malignant_min)
        if label is None:
            continue
        image_path = _resolve_path(row["image"], manifest_path.parent)
        mask_path = _mask_for_row(row, cfg, e2e=e2e, base_dir=manifest_path.parent)
        features = extract_radiomics(image_path, mask_path)
        rows.append(
            {
                "patient_id": str(row["patient_id"]),
                "nodule_id": str(row["nodule_id"]),
                "label": int(label),
                "malignancy_median": float(row["malignancy_median"]),
                "image": str(image_path),
                "mask": str(mask_path),
                **features,
            }
        )

    if not rows:
        raise ValueError("No quedan nódulos benignos/malignos tras descartar malignancy_median == 3")
    table = pd.DataFrame(rows)
    feature_columns = [c for c in table.columns if c not in {"patient_id", "nodule_id", "label", "malignancy_median", "image", "mask"}]
    return {
        "table": table,
        "X": table[feature_columns].to_numpy(dtype=np.float32),
        "y": table["label"].to_numpy(dtype=np.int64),
        "groups": table["patient_id"].to_numpy(dtype=str),
        "feature_names": feature_columns,
    }


def _ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(y_true)
    if total == 0:
        return float("nan")
    ece = 0.0
    for lo, hi in pairwise(bins):
        mask = (y_prob >= lo) & (y_prob < hi if hi < 1.0 else y_prob <= hi)
        if not np.any(mask):
            continue
        confidence = float(np.mean(y_prob[mask]))
        accuracy = float(np.mean(y_true[mask]))
        ece += (float(mask.sum()) / total) * abs(confidence - accuracy)
    return float(ece)


def _safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def _classification_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "auc": _safe_auc(y_true, y_prob),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "ece": _ece(y_true, y_prob),
    }


def _n_splits(y: np.ndarray, groups: np.ndarray, requested: int) -> int:
    unique_groups = np.unique(groups)
    if len(unique_groups) < 2:
        raise ValueError("La CV de clasificación requiere al menos dos grupos de paciente")
    min_class = int(np.min(np.bincount(y.astype(int))))
    n_splits = min(int(requested), len(unique_groups), min_class)
    if n_splits < 2:
        raise ValueError("La CV de clasificación requiere al menos dos muestras por clase")
    return n_splits


def _models(seed: int) -> dict[str, object]:
    models: dict[str, object] = {
        "rf": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed, n_jobs=-1),
        "lasso": LogisticRegression(l1_ratio=1.0, solver="liblinear", class_weight="balanced", max_iter=1000, random_state=seed),
        "mlp": MLPClassifier(hidden_layer_sizes=(32,), alpha=1.0e-3, max_iter=500, random_state=seed),
    }
    try:
        from xgboost import XGBClassifier

        models["xgb"] = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=seed)
    except ImportError:
        pass
    return models


def _make_pipeline(estimator: object, k: int, seed: int) -> Pipeline:
    return Pipeline(steps=[("scale", RobustScaler()), ("select", SelectKBest(partial(mutual_info_classif, random_state=seed), k=k)), ("clf", estimator)])


def _positive_proba(model: Pipeline, X: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    decision = model.decision_function(X)
    return 1.0 / (1.0 + np.exp(-decision))


def evaluate_pipeline(X: np.ndarray, y: np.ndarray, groups: np.ndarray, cfg: Cfg) -> dict:
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    groups = np.asarray(groups)
    if X.ndim != 2:
        raise ValueError(f"X debe ser 2D, shape actual {X.shape}")
    if len(X) != len(y) or len(X) != len(groups):
        raise ValueError("X, y y groups deben tener la misma longitud")

    seed = int(select(cfg, "seed", 42))
    requested_splits = int(select(cfg, "data.n_folds", 5))
    n_splits = _n_splits(y, groups, requested_splits)
    k = min(20, X.shape[1])
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results: dict[str, dict] = {}
    for name, estimator in _models(seed).items():
        oof = np.zeros(len(y), dtype=np.float64)
        fold_rows = []
        for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups=groups)):
            pipe = _make_pipeline(clone(estimator), k=k, seed=seed)
            pipe.fit(X[train_idx], y[train_idx])
            proba = _positive_proba(pipe, X[val_idx])
            oof[val_idx] = proba
            fold_rows.append({"fold": fold, **_classification_metrics(y[val_idx], proba)})
        results[name] = {**_classification_metrics(y, oof), "n_splits": n_splits, "k_features": k, "folds": fold_rows}
    return results


def evaluate_size_only(volumes: np.ndarray, y: np.ndarray, groups: np.ndarray) -> dict:
    volumes = np.asarray(volumes, dtype=np.float32).reshape(-1, 1)
    y = np.asarray(y, dtype=np.int64)
    groups = np.asarray(groups)
    n_splits = _n_splits(y, groups, requested=5)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.zeros(len(y), dtype=np.float64)
    folds = []
    for fold, (train_idx, val_idx) in enumerate(splitter.split(volumes, y, groups=groups)):
        pipe = Pipeline(
            steps=[
                ("scale", RobustScaler()),
                ("clf", LogisticRegression(class_weight="balanced", solver="liblinear", random_state=42)),
            ]
        )
        pipe.fit(volumes[train_idx], y[train_idx])
        proba = pipe.predict_proba(volumes[val_idx])[:, 1]
        oof[val_idx] = proba
        pred = (proba >= 0.5).astype(int)
        folds.append(
            {
                "fold": fold,
                "auc": float(roc_auc_score(y[val_idx], proba)) if len(np.unique(y[val_idx])) == 2 else float("nan"),
                "balanced_acc": float(balanced_accuracy_score(y[val_idx], pred)),
                "brier": float(brier_score_loss(y[val_idx], proba)),
                "ece": _ece(y[val_idx], proba),
            }
        )
    pred = (oof >= 0.5).astype(int)
    return {
        "auc": float(roc_auc_score(y, oof)) if len(np.unique(y)) == 2 else float("nan"),
        "balanced_acc": float(balanced_accuracy_score(y, pred)),
        "brier": float(brier_score_loss(y, oof)),
        "ece": _ece(y, oof),
        "n_splits": n_splits,
        "folds": folds,
    }


def run_phase5(e2e: bool = False) -> dict | None:
    manifest = locate_lidc_manifest()
    if manifest is None:
        print(f"Saltando Phase 5: no existe {LIDC_MANIFEST} ni se encontró nodule_manifest.csv en /kaggle/input.")
        print("Task06 no tiene etiquetas benigno/maligno; Phase 5 requiere LIDC-IDRI.")
        return None

    ensure_phase5_dependencies()
    cfg = make_cfg(experiment_name="phase4_full", outputs=OUTPUTS_ROOT / "phase5", data_name="lidc")
    cfg.data.manifest = str(manifest)
    dataset = build_radiomic_dataset(cfg, e2e=e2e)
    full = evaluate_pipeline(dataset["X"], dataset["y"], dataset["groups"], cfg)
    volumes = dataset["table"].get("original_shape_VoxelVolume")
    if volumes is None:
        volume_candidates = dataset["table"].filter(like="VoxelVolume")
        if volume_candidates.empty:
            raise ValueError("La línea base de tamaño necesita una característica VoxelVolume de PyRadiomics")
        volumes = volume_candidates.iloc[:, 0]
    size_only = evaluate_size_only(volumes.to_numpy(), dataset["y"], dataset["groups"])
    result = {"full": full, "size_only": size_only, "n_nodules": len(dataset["y"])}
    out_path = Path(str(cfg.paths.outputs)) / "classification_results.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print(f"Resultados de clasificación guardados en {out_path}")
    return result


In [ ]:
# Pipeline principal: equivalente a `make pipeline` sin tests/QA.
def run_phase4_all() -> list[dict]:
    summaries = []
    print(f"==> Phase 4 en folds: {PHASE4_FOLDS}")
    for fold in PHASE4_FOLDS:
        print(f"---- Phase 4 fold {fold} ----")
        cfg = make_cfg(experiment_name="phase4_full", fold=fold, outputs=OUTPUTS_ROOT / "phase4" / f"fold_{fold}")
        summaries.append(train_segmentation_run(cfg))
    return summaries


def run_phase6_all() -> list[dict]:
    results = []
    print(f"==> Phase 6 ablation en folds: {PHASE6_FOLDS}")
    for fold in PHASE6_FOLDS:
        fold_out = OUTPUTS_ROOT / "phase6" / f"fold_{fold}"
        for frac in FRACTIONS:
            for aug in AUGS:
                for seed in SEEDS:
                    print(f"---- Phase 6 fold={fold} fraction={frac} aug={aug} seed={seed} ----")
                    cfg = make_cfg(experiment_name="phase6_ablation", fold=fold, outputs=fold_out, seed=seed)
                    update_cfg(cfg, "data_fraction", float(frac))
                    update_cfg(cfg, "aug_regime", str(aug))
                    result = run_ablation_cell(cfg)
                    report = analyze_ablation(fold_out)
                    print(f"Ablation cell: {result['result_path']}")
                    print(f"Ablation report: {report}")
                    results.append(result)
    return results


def print_summaries(root: Path) -> None:
    print(f"==> Summaries en {root}")
    if not root.exists():
        print(f"No existe {root}.")
        return
    found = False
    for path in sorted(root.rglob("summary.json")):
        found = True
        print(f"\n---- {path} ----")
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(payload, indent=2))
    if not found:
        print("No hay summary.json todavía.")


OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Pipeline preparado. Ejecutando fases activas...")
print("Nota: el sweep completo puede tardar mucho: Phase 4 son varios folds y Phase 6 es folds x fracciones x aumentos x semillas.")

phase4_summaries = run_phase4_all() if RUN_PHASE4 else []
phase6_results = run_phase6_all() if RUN_PHASE6 else []
phase5_result = run_phase5(e2e=PHASE5_E2E) if RUN_PHASE5 else None
print_summaries(OUTPUTS_ROOT)
print("Pipeline completado.")
